# Paridade de engine: masterclass `main` × `release/v0.5` (issue #91)

Mesma história de negócio (trocar `legacy_score` por `score_5`, cutoffs regionais, rating A–E),
rodada **passo a passo** nas duas engines, cada uma no seu venv/worktree, em subprocesso.
Este notebook **não importa `pycreditools`** — só carrega os JSONs que `measure_main.py` e
`measure_v05.py` emitiram (regenere tudo com `run_all.ps1` / `run_all.sh`).

Regras do experimento (**protocolo de condições iguais**):

- **Toda métrica sai de `policy.simulate(...).data`** — nenhum fast-path de sweep.
- Contrato de métrica único nos dois lados (ADR 0008, fórmula crua do funil):
  aprovação pré take-up, inad ponderada por contratado.
- Hard filters do challenger **fixados** no conjunto que o sugestor da v0.5 escolheu
  (`vl_negativacao<=0 & vl_vencido_scr<=0 & vl_protestos<=0`) — usado igual nos dois lados.
- **`calibration_bins=5` nas DUAS engines** (o parâmetro existe nas duas) e
  **stress ×1,5 SEMPRE**: toda inad de challenger reportada como manchete é a estressada
  ×1,5 — a mesma premissa nos dois lados. As variantes sem stress, ×1,3 e o oráculo
  `true_pd` continuam nas tabelas como colunas de diagnóstico.
- **Modo A**: cada engine gera seus dados (`generate_sample_data(seed=7)`).
  **Modo B**: as duas leem o MESMO parquet (`shared_base.parquet`, gerador v0.5 + colunas
  derivadas que a `main` pede). Uma cópia em Excel (`shared_base.xlsx`) fica disponível
  para conferência manual.


In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

R = Path("results")
runs = {name: json.loads((R / f"{name}.json").read_text(encoding="utf-8"))
        for name in ["main_A_60k", "v05_A_60k", "main_B_60k", "v05_B_60k",
                     "main_A_20k", "v05_A_20k"]}
mA, vA = runs["main_A_60k"], runs["v05_A_60k"]
mB, vB = runs["main_B_60k"], runs["v05_B_60k"]

def side_by_side(rows, main_run=None, v05_run=None, digits=4):
    """rows: list of (label, fn(run)). Returns main × v0.5 × diff table."""
    main_run = main_run or mA
    v05_run = v05_run or vA
    out = []
    for label, fn in rows:
        a, b = fn(main_run), fn(v05_run)
        row = {"passo": label, "main": a, "v0.5": b}
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            row["diff (v0.5 − main)"] = round(b - a, digits)
        out.append(row)
    return pd.DataFrame(out).set_index("passo")


## Fatos de engine (o que cada branch tem)

Antes dos números: as diferenças de API que os scripts detectaram por introspecção.


In [2]:
facts = ["package_version", "rate_has_observed_col", "optimize_has_directions",
         "has_suggest_hard_filters", "legacy_quantile_exported",
         "calibration_bins_supported", "actual_default_masked_frac"]
pd.DataFrame({"main": {f: mA["engine_facts"][f] for f in facts},
              "v0.5": {f: vA["engine_facts"][f] for f in facts}})


,main,v0.5
package_version,n/a,n/a
rate_has_observed_col,False,True
optimize_has_directions,False,True
has_suggest_hard_filters,False,True
legacy_quantile_exported,False,True
calibration_bins_supported,True,True
actual_default_masked_frac,0.9047,0.9047


## Passo 0 — Os dados são os mesmos? (causa-raiz nº 1)

A issue supunha que os geradores divergem para o mesmo seed. **Medido: não divergem.**
Para `seed=7`, as colunas compartilhadas dos dois geradores são **byte a byte idênticas**
(a v0.5 sorteia `passed_antifraud`/`market_default`/`sample` *depois* do stream comum, então
não desloca o RNG). A prova operacional está abaixo: para cada engine, o modo A (dados
próprios) e o modo B (parquet compartilhado) produzem exatamente as mesmas métricas.

> **Veredito causa nº 1 (dados): descartada.** Todo o desencontro é engine/premissa, não dados.


In [3]:
sections = ["incumbent", "ks_on_hf_approved", "frontier", "champion_iso_approval",
            "three_policies", "regional_cutoffs", "rating", "swap_in_calibration",
            "full_funnel"]
checks = []
for branch, a_run, b_run in [("main", mA, mB), ("v0.5", vA, vB)]:
    row = {"engine": branch}
    for s in sections:
        row[f"{s} (A ≡ B)"] = a_run[s] == b_run[s]
    checks.append(row)
pd.DataFrame(checks).set_index("engine").T


engine,main,v0.5
incumbent (A ≡ B),True,True
ks_on_hf_approved (A ≡ B),True,True
frontier (A ≡ B),True,True
champion_iso_approval (A ≡ B),True,True
three_policies (A ≡ B),True,True
regional_cutoffs (A ≡ B),True,True
rating (A ≡ B),True,True
swap_in_calibration (A ≡ B),True,True
full_funnel (A ≡ B),True,True


Como A ≡ B, daqui em diante todas as tabelas usam o **modo A a 60k** (o número
“de verdade” que cada lado publicaria) — e ele coincide com o modo B.

## Passo 1 — Incumbente (HF de entrada + cutoff no `legacy_score` q0.78 + take-up)

Única diferença de código: o estágio de take-up.
`main`: `.rate("Take-up", base_rate=1.0, variable="conversion_rate")` (coluna de propensão).
`v0.5`: `.rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")` (outcome observado).


In [4]:
side_by_side([
    ("aprovação (pré take-up, ADR 0008)", lambda r: r["incumbent"]["approval"]),
    ("inad (ponderada por contratado)",   lambda r: r["incumbent"]["default"]),
    ("contratados",                        lambda r: r["incumbent"]["contracted"]),
    ("take-up",                            lambda r: r["incumbent"]["take_up"]),
    ("inad real (oráculo true_pd)",        lambda r: r["incumbent"]["default_true"]),
    ("aprovação pós take-up (contrato antigo)",
        lambda r: r["incumbent"]["legacy_contract"]["approval_post_take_up"]),
    ("inad não ponderada (contrato antigo)",
        lambda r: r["incumbent"]["legacy_contract"]["default_unweighted"]),
])


,main,v0.5,diff (v0.5 − main)
passo,,,
"aprovação (pré take-up, ADR 0008)",0.2055,0.2055,0.0000
inad (ponderada por contratado),0.0754,0.0754,0.0000
contratados,"5,719.0000","5,719.0000",0.0000
take-up,0.4639,0.4639,0.0000
inad real (oráculo true_pd),0.0707,0.0707,0.0000
aprovação pós take-up (contrato antigo),0.0953,0.0953,0.0000
inad não ponderada (contrato antigo),0.0754,0.0754,0.0000


**Leitura.** Idênticos até o 4º decimal, inclusive contratados (5.719). No incumbente não
há swap-in (a política reproduz a aprovação vigente), então os dois idiomas de take-up
convergem para o volume observado — e batem com a referência da issue (20,5% / 7,5% / 5.719).

Também recomputamos o **contrato antigo** de métrica nos dois lados: mesmo número nos dois.
> **Veredito causa nº 2 (contrato de métrica): não gera gap numérico** quando a mesma fórmula
> é aplicada aos mesmos dados. O ADR 0008 muda *rótulo e denominador reportados*, não o funil.
> Diferenças de manchete README × masterclass vêm de qual fórmula cada texto escolheu exibir.

## Passo 2 — Hard filters do challenger (causa-raiz nº 5)

Na v0.5 rodamos `suggest_hard_filters` (bad_col=`actual_default`, floor 0,55, lift ≥ 1,3);
na `main` o sugeridor não existe. O conjunto sugerido foi **exatamente** o conjunto que
fixamos nos dois lados — ou seja, a seleção de HF não pode explicar diferença nenhuma aqui.


In [5]:
hf_v, hf_m = vA["hard_filters"], mA["hard_filters"]
print("regras fixadas (ambas engines):", hf_m["used_rules"])
print("taxa de passagem dos HF:        ", f"{hf_m['hf_pass_rate']:.4f} (idêntica nos dois lados)")
print()
print("sugestor v0.5 escolheu:", hf_v["suggested"]["rules"])
print("orçamento do sugestor: ", {k: round(v, 3) for k, v in hf_v["suggested"]["budget"].items()})


regras fixadas (ambas engines): ['vl_negativacao lte 0', 'vl_vencido_scr lte 0', 'vl_protestos lte 0']
taxa de passagem dos HF:         0.5769 (idêntica nos dois lados)

sugestor v0.5 escolheu: ['vl_negativacao lte 0', 'vl_vencido_scr lte 0', 'vl_protestos lte 0']
orçamento do sugestor:  {'floor': 0.55, 'spent': 0.423, 'approval_rate': 0.577, 'headroom': 0.027}


> **Veredito causa nº 5 (seleção de HF): descartada por construção** — e o sugestor,
> deixado livre, escolhe o mesmo conjunto que o README usa à mão.

## Passo 3 — KS por score, só sobre os aprovados pelos HF

`ModelEvaluator.compute_ks` sobre `actual_default`, subpopulação `HF == True`, nos dois lados.


In [6]:
ks = pd.DataFrame({"main": mA["ks_on_hf_approved"], "v0.5": vA["ks_on_hf_approved"]})
ks["diff"] = ks["v0.5"] - ks["main"]
ks.sort_values("v0.5", ascending=False)


,main,v0.5,diff
score_5,0.3333,0.3333,0.0000
score_4,0.3196,0.3196,0.0000
score_3,0.2975,0.2975,0.0000
score_2,0.2947,0.2947,0.0000
legacy_score,0.2757,0.2757,0.0000


**Leitura.** KS idêntico (score_5 = 0,3333, batendo a referência da issue). Ranking igual;
`score_5` vence nos dois lados. Nenhuma divergência de avaliação de modelo.

## Passos 4–5 — Challenger `score_5`: fronteira e política iso-aprovação

Grade de cutoffs idêntica (quantis do `score_5`), cada ponto simulado com `.simulate()`,
**`calibration_bins=5` e stress ×1,5 nos dois lados** (condições iguais — o parâmetro existe
nas duas engines; o `CalibrationReliabilityWarning` que motiva o bins=5 veio no PR #72).


In [7]:
ch = lambda r: r["champion_iso_approval"]
side_by_side([
    ("cutoff escolhido",                     lambda r: ch(r)["cutoff"]),
    ("aprovação",                             lambda r: ch(r)["approval"]),
    ("contratados",                           lambda r: ch(r)["contracted"]),
    ("INAD MANCHETE (stress ×1,5)",           lambda r: ch(r)["default"]),
    ("inad stress ×1,5 MEDIDA (policy.stress)", lambda r: ch(r)["default_stress_1.5_measured"]),
    ("inad sem stress (diagnóstico)",         lambda r: ch(r)["default_nostress"]),
    ("inad stress ×1,3 (diagnóstico)",        lambda r: ch(r)["default_stress_1.3"]),
    ("inad real (oráculo true_pd)",           lambda r: ch(r)["default_true"]),
    ("inad do incumbente (referência)",       lambda r: r["incumbent"]["default"]),
])


,main,v0.5,diff (v0.5 − main)
passo,,,
cutoff escolhido,736.0000,736.0000,0.0000
aprovação,0.1992,0.1992,0.0000
contratados,"5,526.0500","5,573.6907",47.6407
"INAD MANCHETE (stress ×1,5)",0.0690,0.0695,0.0005
"inad stress ×1,5 MEDIDA (policy.stress)",0.0690,0.0695,0.0005
inad sem stress (diagnóstico),0.0566,0.0568,0.0002
"inad stress ×1,3 (diagnóstico)",0.0640,0.0644,0.0004
inad real (oráculo true_pd),0.0639,0.0644,0.0005
inad do incumbente (referência),0.0754,0.0754,0.0000


**Leitura, com carinho:**

- **Sob condições iguais, as engines colam.** Mesmo corte (736), manchete estressada
  **6,90% (main) vs 6,95% (v0.5)** — 0,05 p.p. de resíduo, todo explicado pelo idioma de
  take-up (volumes contratados 5.526 vs 5.574). A linha “medida” confirma que a manchete
  derivada é exatamente o que `policy.stress(1.5)` entrega no caminho de código real.
- **Mesmo com a premissa conservadora ×1,5, o challenger ganha**: 6,90–6,95% contra 7,54%
  do incumbente na mesma aprovação (−0,6 p.p.). Contra o oráculo o ganho é maior
  (6,4% vs 7,5%, −1,1 p.p.) — o ×1,5 come metade do ganho aparente, porque exagera o PD
  do swap-in (dial abaixo).
- As colunas de diagnóstico mostram o dial completo: sem stress 5,7% (otimista), ×1,3
  6,4% (≈ oráculo), ×1,5 6,9–7,0% (conservador).

### Quadrantes no cutoff 736


In [8]:
quads = []
for scen in ["keep_in", "swap_in", "swap_out", "keep_out"]:
    qm = ch(mA)["quadrants"].get(scen, {})
    qv = ch(vA)["quadrants"].get(scen, {})
    quads.append({
        "quadrante": scen,
        "n (main)": qm.get("n"), "n (v0.5)": qv.get("n"),
        "vol contratado (main)": qm.get("vol"), "vol contratado (v0.5)": qv.get("vol"),
        "PD sim (main)": qm.get("sim"), "PD sim (v0.5)": qv.get("sim"),
        "PD real (main)": qm.get("true"), "PD real (v0.5)": qv.get("true"),
        "inad observada (main)": qm.get("actual"), "inad observada (v0.5)": qv.get("actual"),
    })
pd.DataFrame(quads).set_index("quadrante")


,n (main),n (v0.5),vol contratado (main),vol contratado (v0.5),PD sim (main),PD sim (v0.5),PD real (main),PD real (v0.5),inad observada (main),inad observada (v0.5)
quadrante,,,,,,,,,,
keep_in,8301,8301,"3,715.0000","3,715.0000",0.0471,0.0471,0.0460,0.0460,0.0471,0.0471
swap_in,3653,3653,"1,811.0500","1,858.6907",0.0760,0.0762,0.1005,0.1010,NaN,NaN
swap_out,4027,4027,0.0000,0.0000,NaN,NaN,NaN,NaN,0.1277,0.1277
keep_out,44019,44019,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN


**Leitura.**

- As **populações são idênticas** (mesmos n por quadrante — consequência do passo 0).
- **Causa nº 3 (take-up)**: o volume contratado de swap-in difere — 1.811 (main, coluna de
  propensão `conversion_rate`) vs 1.859 (v0.5, `observed_col="hired"` calibrado por score),
  +2,6%. Efeito pequeno mas real no denominador da inad: 5.526 vs 5.574 contratados no total.
- **Causa nº 4 (PD do swap-in)**: PD simulado do swap-in 8,16% (main) vs 7,62% (v0.5-bins5),
  contra **10,1% real**. As duas subestimam; a v0.5 com bins=5 subestima *mais* — ver dial.
- swap-out idêntico (12,77% observado): quem o challenger corta é igual nos dois lados.

### O dial do swap-in PD (causa-raiz nº 4, o coração do desencontro)


In [9]:
sc_m, sc_v = mA["swap_in_calibration"], vA["swap_in_calibration"]
ch_m, ch_v = ch(mA), ch(vA)
dial = pd.DataFrame([
    {"medida": "imputado (bins=5, o protocolo)", "main": sc_m["swap_in_pd_imputed"],
     "v0.5": sc_v["swap_in_pd_imputed"]},
    {"medida": "imputado (decis default — contrafactual)",
     "main": sc_m.get("swap_in_pd_imputed_deciles"),
     "v0.5": sc_v.get("swap_in_pd_imputed_deciles")},
    {"medida": "imputado × stress 1,3 (derivado)", "main": sc_m["swap_in_pd_stress_1.3"],
     "v0.5": sc_v["swap_in_pd_stress_1.3"]},
    {"medida": "imputado × stress 1,5 (derivado)", "main": sc_m["swap_in_pd_stress_1.5"],
     "v0.5": sc_v["swap_in_pd_stress_1.5"]},
    {"medida": "stress 1,5 MEDIDO via policy.stress(1.5)",
     "main": ch_m["swap_in_pd_stress_1.5_measured"],
     "v0.5": ch_v["swap_in_pd_stress_1.5_measured"]},
    {"medida": "REAL (oráculo true_pd)", "main": sc_m["swap_in_pd_true"],
     "v0.5": sc_v["swap_in_pd_true"]},
]).set_index("medida")
dial


,main,v0.5
medida,,
"imputado (bins=5, o protocolo)",0.0760,0.0762
imputado (decis default — contrafactual),0.0816,0.0813
"imputado × stress 1,3 (derivado)",0.0988,0.0991
"imputado × stress 1,5 (derivado)",0.1140,0.1143
"stress 1,5 MEDIDO via policy.stress(1.5)",0.1140,0.1143
REAL (oráculo true_pd),0.1005,0.1010


**Leitura.**

- **No protocolo (bins=5), as engines coincidem**: 7,60% vs 7,62%. No contrafactual
  (decis default) idem: 8,16% vs 8,13%. O resíduo de ~0,03 p.p. é só o peso de take-up.
  **Não há bug de calibração entre engines** — dado o mesmo knob, mesma resposta.
- Contra o real (10,1%), o imputado cru subestima ~25%. **Markup honesto ≈ ×1,3**:
  7,6 × 1,3 = 9,9% ≈ real. **×1,5 superestima** (11,4% vs 10,1%) — é o mecanismo que fazia
  o “modelo melhor” parecer sem ganho; sob condições iguais ele come ~metade do ganho
  verdadeiro, mas não o apaga (tabela anterior).
- A linha “stress 1,5 MEDIDO” veio de um `.simulate()` com `policy.stress(1.5)` real em cada
  engine — coincide com a derivação, validando `AggravationStress = clip(pd × fator)` no
  caminho de código de verdade, nos dois lados.

> **Critério de aceite da issue:** confirmado que o “sem ganho” era artefato do ×1,5 — contra
> `true_pd` o ganho é −1,1 p.p. e com ×1,3 a inad simulada cola no real. Sob o protocolo
> ×1,5 fixado nos dois lados, ainda sobra ganho de −0,6 p.p., idêntico nas duas engines.

### Três políticas na fronteira (localizadas pela inad ESTRESSADA ×1,5, o protocolo)


In [10]:
pols = []
for pm, pv in zip(mA["three_policies"], vA["three_policies"]):
    pols.append({
        "política": pm["name"],
        "cutoff (main)": pm["cutoff"], "cutoff (v0.5)": pv["cutoff"],
        "aprovação (main)": pm["approval"], "aprovação (v0.5)": pv["approval"],
        "inad ×1,5 (main)": pm["default"], "inad ×1,5 (v0.5)": pv["default"],
        "inad real (main)": pm["default_true"], "inad real (v0.5)": pv["default_true"],
        "contratados (main)": pm["contracted"], "contratados (v0.5)": pv["contracted"],
    })
pd.DataFrame(pols).set_index("política")


,cutoff (main),cutoff (v0.5),aprovação (main),aprovação (v0.5),"inad ×1,5 (main)","inad ×1,5 (v0.5)",inad real (main),inad real (v0.5),contratados (main),contratados (v0.5)
política,,,,,,,,,,
iso_approval,736,736,0.1992,0.1992,0.0690,0.0695,0.0639,0.0644,"5,526.0500","5,573.6907"
iso_default,736,736,0.1992,0.1992,0.0690,0.0695,0.0639,0.0644,"5,526.0500","5,573.6907"
balanced,736,736,0.1992,0.1992,0.0690,0.0695,0.0639,0.0644,"5,526.0500","5,573.6907"


**Leitura.**

- **Sob condições iguais, as três políticas colapsam pro MESMO corte (736) nos dois lados.**
  Com o stress ×1,5 no protocolo, nenhum corte da grade expande aprovação além do incumbente
  mantendo inad estressada ≤ 7,54% — iso-aprovação, iso-inad e equilibrada viram a mesma
  política. Leitura de negócio honesta: sob ×1,5, o `score_5` compra **redução de risco na
  mesma aprovação** (−0,6 p.p.), não expansão de aprovação.
- Contra o oráculo, a folga é maior (6,4% vs 7,54%): a premissa ×1,5 deixa ~0,5 p.p. de
  ganho na mesa. É o preço declarado do conservadorismo — igual nos dois lados.

## Passo 6 — Cutoffs regionais vs corte geral (causa-raiz nº 6)

Alvo comum: a inad do incumbente (7,54%). Corte geral e cortes por região escolhidos pela
**inad estressada ×1,5** (o protocolo) de cada engine — mesma grade, mesmo critério.


In [11]:
reg_rows = []
reg_m = {r["region"]: r for r in mA["regional_cutoffs"]["regions"]}
reg_v = {r["region"]: r for r in vA["regional_cutoffs"]["regions"]}
for region in sorted(reg_m):
    a, b = reg_m[region], reg_v[region]
    reg_rows.append({"região": region, "n": a["applicants"],
                     "cutoff (main)": a["cutoff"], "cutoff (v0.5)": b["cutoff"],
                     "aprovação (main)": a["approval"], "aprovação (v0.5)": b["approval"],
                     "inad ×1,5 (main)": a["default"], "inad ×1,5 (v0.5)": b["default"]})
display(pd.DataFrame(reg_rows).set_index("região"))

tot = []
for name, key in [("corte geral", "general"), ("segmentado (soma das regiões)", "segmented_total")]:
    gm, gv = mA["regional_cutoffs"][key], vA["regional_cutoffs"][key]
    tot.append({"política": name,
                "aprovação (main)": gm["approval"], "aprovação (v0.5)": gv["approval"],
                "inad ×1,5 (main)": gm["default"], "inad ×1,5 (v0.5)": gv["default"],
                "contratados (main)": gm["contracted"], "contratados (v0.5)": gv["contracted"]})
pd.DataFrame(tot).set_index("política")


,n,cutoff (main),cutoff (v0.5),aprovação (main),aprovação (v0.5),"inad ×1,5 (main)","inad ×1,5 (v0.5)"
região,,,,,,,
Centro-Oeste,5970,752,752,0.1811,0.1811,0.0727,0.0739
Nordeste,10812,785,785,0.1499,0.1499,0.0623,0.0627
Norte,4125,732,732,0.1840,0.1840,0.0739,0.0740
Sudeste,26989,694,694,0.2308,0.2308,0.0735,0.0740
Sul,12104,680,680,0.2500,0.2500,0.0727,0.0726


,aprovação (main),aprovação (v0.5),"inad ×1,5 (main)","inad ×1,5 (v0.5)",contratados (main),contratados (v0.5)
política,,,,,,
corte geral,0.1992,0.1992,0.0690,0.0695,"5,526.0500","5,573.6907"
segmentado (soma das regiões),0.2119,0.2119,0.0719,0.0722,"5,970.8500","6,006.5674"


**Leitura.**

- **Cortes regionais IDÊNTICOS nos dois lados** (Centro-Oeste 752, Nordeste 785, Norte 732,
  Sudeste 694, Sul 680). Nas rodadas anteriores (sem condições iguais) eles divergiam até
  105 pontos — a divergência era 100% premissa de calibração/stress, zero engine.
- Segmentada vs geral: +1,3 p.p. de aprovação (21,2% vs 19,9%) mantendo a inad estressada
  dentro do alvo, nas duas engines. O delta de contratados entre engines (5.971 vs 6.007)
  é o idioma de take-up, como sempre.
- O “5,80% de PD estressado” do README continua não reproduzível pelo fluxo canônico: alvo
  escolhido à mão com outra premissa. Diferença de *setup*, não de engine.

### Bônus do passo 6 — o `optimize_cutoffs` de cada engine, checado contra `.simulate()`

A comparação acima usou a mesma grade de `.simulate()` nos dois lados (regra de ouro).
Aqui deixamos **cada engine usar seu próprio otimizador** (como a masterclass e o README
fazem) com o mesmo alvo, e re-reportamos as métricas do corte escolhido via `.simulate()`.


In [12]:
oc = []
for tag, r in [("main", mA), ("v0.5", vA)]:
    o = r["optimizer_check"]
    oc.append({"engine": tag, "cutoff escolhido": o["cutoff"],
               "inad alegada pelo otimizador": o["optimizer_metrics"]["overall_default_rate"],
               "inad re-simulada (.simulate)": o["resimulated"]["default"],
               "inad real (oráculo)": o["resimulated"]["default_true"],
               "aprovação": o["resimulated"]["approval"],
               "alvo": r["regional_cutoffs"]["target_default"]})
pd.DataFrame(oc).set_index("engine")


,cutoff escolhido,inad alegada pelo otimizador,inad re-simulada (.simulate),inad real (oráculo),aprovação,alvo
engine,,,,,,
main,688,0.0723,0.0845,0.0778,0.2306,0.0754
v0.5,751,0.0724,0.0625,0.0602,0.1892,0.0754


**Leitura — o achado mais operacional do estudo.**

- Mesmo sob condições iguais (bins=5 + stress ×1,5 na config dos dois), os otimizadores
  **discordam entre si e de si mesmos**: a **main** alega inad 7,2% no corte 688, mas o
  `.simulate()` estressado no mesmo corte dá **8,5%** — fura o alvo de 7,54% em ~1 p.p.
  (fast-path otimista/inseguro). A **v0.5** alega 7,2% no corte 751 e o `.simulate()`
  estressado dá **6,3%** — conservadora (a inflação de swap-in do sweep que a issue #91
  já tinha flagrado); deixa aprovação na mesa, mas não fura alvo.
- É a única etapa onde igualar premissas NÃO fecha o gap — a diferença está no código do
  fast-path, não na configuração. Regra de ouro confirmada: **fast-path localiza,
  `.simulate()` reporta.**

## Passo 7 — Rating A–E e validação DEV/OOT

`fit_risk_groups(score_5, actual_default, bins=20, max_groups=5, min_vol_ratio=0.05,
max_crossings=3, time_col="safra", oot_date="2025-01")` sobre os contratados, nos dois lados.


In [13]:
rt = []
rm = {(r["grade"], r["period"]): r["pd"] for r in mA["rating"]["rows"]}
rv = {(r["grade"], r["period"]): r["pd"] for r in vA["rating"]["rows"]}
for grade in "ABCDE":
    rt.append({"grade": grade,
               "PD Train (main)": rm.get((grade, "Train")), "PD Train (v0.5)": rv.get((grade, "Train")),
               "PD OOT (main)": rm.get((grade, "OOT")), "PD OOT (v0.5)": rv.get((grade, "OOT"))})
pd.DataFrame(rt).set_index("grade")


,PD Train (main),PD Train (v0.5),PD OOT (main),PD OOT (v0.5)
grade,,,,
A,0.0138,0.0138,0.0239,0.0239
B,0.0525,0.0525,0.0807,0.0807
C,0.0781,0.0781,0.0782,0.0782
D,0.1240,0.1240,0.1123,0.1123
E,0.1757,0.1757,0.1769,0.1769


**Leitura.** Grades e PDs **idênticos** (5 grupos, amplitude Train 1,4% → 17,6%).
O motor de rating não mudou entre as branches.

## Stress de estágios — funil completo: antifraude (taxa fixa) + conversão por score

Dois RateStages empilhados no corte iso-aprovação: **antifraude risk-independent**
(main: `base_rate=0.90` flat; v0.5: `observed_col="passed_antifraud"`, `calibrate_by=None`,
taxa observada ~0,90) e **conversão que discrimina por score** — quanto pior o score,
maior a conversão (main: coluna `conversion_rate = 0.95 − decil×0.06`; v0.5:
`observed_col="hired"` calibrado por score, mesma forma por construção do gerador).

O que uma engine sã precisa entregar aqui: aprovação (pré-rate) **intacta**, inad
**quase invariante** sob o estágio fixo, volume contratado escalando pela taxa, e o
gradiente de conversão decrescente no score.


In [14]:
fm, fv = mA["full_funnel"], vA["full_funnel"]
side_by_side([
    ("aprovação (deve ficar igual ao 1 estágio)", lambda r: r["full_funnel"]["approval"]),
    ("aprovação com 1 estágio (referência)",
        lambda r: r["full_funnel"]["reference_single_stage"]["approval"]),
    ("contratados (2 estágios)", lambda r: r["full_funnel"]["contracted"]),
    ("contratados (1 estágio)",
        lambda r: r["full_funnel"]["reference_single_stage"]["contracted"]),
    ("razão contratados 2/1 estágios",
        lambda r: r["full_funnel"]["contracted"]
        / r["full_funnel"]["reference_single_stage"]["contracted"]),
    ("inad (2 estágios)", lambda r: r["full_funnel"]["default"]),
    ("inad (1 estágio)", lambda r: r["full_funnel"]["reference_single_stage"]["default"]),
    ("inad real (oráculo)", lambda r: r["full_funnel"]["default_true"]),
    ("ordem invertida ≡ (take-up antes do antifraude)",
        lambda r: r["full_funnel"]["order_invariant"]),
])


,main,v0.5,diff (v0.5 − main)
passo,,,
aprovação (deve ficar igual ao 1 estágio),0.1992,0.1992,0.0000
aprovação com 1 estágio (referência),0.1992,0.1992,0.0000
contratados (2 estágios),"5,344.9450","4,990.2776",-354.6674
contratados (1 estágio),"5,526.0500","5,573.6907",47.6407
razão contratados 2/1 estágios,0.9672,0.8953,-0.0719
inad (2 estágios),0.0675,0.0697,0.0022
inad (1 estágio),0.0690,0.0695,0.0005
inad real (oráculo),0.0626,0.0650,0.0024
ordem invertida ≡ (take-up antes do antifraude),True,True,0.0000


In [15]:
grad = pd.DataFrame({
    "take-up main": fm["take_up_by_score_quintile"],
    "take-up v0.5": fv["take_up_by_score_quintile"],
})
grad.index.name = "quintil do score_5 (q1 = pior score)"
grad


,take-up main,take-up v0.5
quintil do score_5 (q1 = pior score),,
q1,0.4969,0.4948
q2,0.4719,0.4467
q3,0.4479,0.4085
q4,0.4011,0.3611
q5,0.4164,0.3745


**Leitura.**

- **Aprovação pré-rate intacta** nos dois lados (rate stages não mexem no funil de
  aprovação — contrato ADR 0008 respeitado pelas duas engines).
- **Gradiente correto nos dois lados**: conversão cai do pior quintil (q1 ≈ 0,50) pro melhor
  (q4–q5 ≈ 0,36–0,42, com ruído amostral no topo) — o “quanto pior o score, maior a
  conversão” pedido, nas duas engines.
- **A diferença que importa está nos keep-ins.** O contratado da main cai menos que ×0,90:
  a main dá **bypass total (probs=1,0) em qualquer rate stage não-conversão** pros já
  aprovados (`stages.py`, heurística por nome/posição — “conversão” = `calibrate=True`,
  nome em `{conversao, conversion, hired, take_up, take_up_rate}` ou o *último* RateStage).
  Antifraude fixo nunca toca o livro histórico. A v0.5 aplica o `passed_antifraud`
  **observado** (~0,90) nos keep-ins — comportamento **declarativo** (`observed_col`
  presente → usa observado; ausente → 1,0), sem mágica de nome.
- São **premissas de dado** distintas, ambas defensáveis (main: “quem foi aprovado e
  contratou historicamente já passou no antifraude”; v0.5: “a coluna observada existe e
  não está dobrada em `approved`/`hired`” — que é como o gerador da v0.5 define
  `passed_antifraud`). Mas a da main é **implícita e frágil**: depende do nome/ordem do
  estágio. Renomear “Take-up” para algo fora da lista mágica, ou empilhar um terceiro
  RateStage, muda silenciosamente qual estágio ganha o `hired` no keep-in.
- O probe de ordem invertida deu ≡ nos dois lados — na v0.5 por design (produto
  comutativo); na main por sorte aritmética (o bypass entrega `hired × 1,0` nas duas
  ordens e `hired` é 0/1, então o produto não muda). Com um estágio de rate *fracionário*
  no lugar de `hired` a coincidência quebraria.

## Sensibilidade a tamanho: n = 20.000 (default da `main`)


In [16]:
m20, v20 = runs["main_A_20k"], runs["v05_A_20k"]
sens = []
for tag, r60, r20 in [("main", mA, m20), ("v0.5", vA, v20)]:
    for n_lbl, r in [("60k", r60), ("20k", r20)]:
        c = r["champion_iso_approval"]
        sens.append({"engine": tag, "n": n_lbl,
                     "inad incumbente": r["incumbent"]["default"],
                     "challenger manchete (×1,5)": c["default"],
                     "challenger sem stress": c["default_nostress"],
                     "challenger real (oráculo)": c["default_true"]})
pd.DataFrame(sens).set_index(["engine", "n"])


inad incumbente  challenger manchete (×1,5)  challenger sem stress  challenger real (oráculo)
engine n                                                                                                 
main   60k           0.0754                      0.0690                 0.0566                     0.0639
       20k           0.0712                      0.0655                 0.0530                     0.0695
v0.5   60k           0.0754                      0.0695                 0.0568                     0.0644
       20k           0.0712                      0.0652                 0.0529                     0.0693

**Leitura.** Em 20k o veredito não muda: engines coladas entre si e challenger melhor
que o incumbente contra o oráculo, com margem menor — coerente com uma base 3× menor.
Nenhuma conclusão deste estudo depende do tamanho da amostra.

## Warnings capturados (`notes`)

A v0.5 emite `CalibrationReliabilityWarning` (PR #72) durante a varredura — capturado, não
suprimido. A `main` emite só o aviso genérico de decis.


In [17]:
import textwrap
for tag, r in [("main", mA), ("v0.5", vA)]:
    uniq = []
    for n_ in r["notes"]:
        key = n_[:80]
        if key not in [u[:80] for u in uniq]:
            uniq.append(n_)
    print(f"— {tag}: {len(r['notes'])} warnings, {len(uniq)} distintos")
    for u in uniq:
        print(textwrap.fill(u, 110, initial_indent="   • ", subsequent_indent="     "))
    print()


— main: 1 warnings, 1 distintos
   • [UserWarning] Swap-in PD imputation is using the default of 10 score bins (deciles). Pass
     CreditPolicy(calibration_bins=...) to set a different granularity.

— v0.5: 28 warnings, 28 distintos
   • [UserWarning] 'actual_default' is observed for 9.5% of the base; every lift is measured among the
     contracted, a population the incumbent policy already selected. See the measurement caveat in
     suggest_hard_filters' docstring.
   • [CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 10% of swap-ins score outside the
     keep-ins' observed range, so their imputed PD is edge-clamp extrapolation, not measurement. No bin count
     fixes this — the score range has no observed defaults to measure. Pass an estimated_default_col with a
     model PD.
   • [CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 6% of swap-ins score outside the
     keep-ins' observed range, so their imputed PD is edge-clamp extrapolati

## Veredito por causa-raiz

| # | Causa candidata | Veredito | Evidência |
|---|---|---|---|
| 1 | Dados diferentes pro mesmo seed | **Descartada** | Colunas compartilhadas byte a byte idênticas; modo A ≡ modo B nas duas engines (passo 0) |
| 2 | Contrato de métrica (ADR 0008) | **Sem gap numérico** | Mesma fórmula nos mesmos dados → mesmos números; contrato antigo recomputado também coincide (passo 1) |
| 3 | Take-up (propensão × observed_col) | **Esperada, pequena — o único resíduo** | +2,6% de volume swap-in na v0.5 (1.859 vs 1.811); mecanismo do ADR 0008 / RateStage genérico (#68); explica os ~0,05 p.p. que sobram em toda tabela |
| 4 | Swap-in PD + stress | **Fechada sob condições iguais** | Mesmo knob → mesma resposta: bins=5 dá 7,60% vs 7,62%, decis dá 8,16% vs 8,13%; real 10,1%; ×1,3 honesto, ×1,5 exagera (mas fixado nos dois, ainda sobra −0,6 p.p. de ganho) |
| 5 | Seleção de HF | **Descartada** | Sugestor (#73) escolhe exatamente o conjunto fixado; HF pass rate idêntica |
| 6 | Cutoffs regionais | **Fechada sob condições iguais** | Cortes regionais idênticos nas 5 regiões; a divergência anterior era premissa, não engine. Exceção: o fast-path do `optimize_cutoffs` (main inseguro, v0.5 conservador) — diferença de código, não de config |

Detalhes e recomendação de merge: `validation/README.md`.
